<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/W6D4_Daily_Challenge_Finetune_LLM_with_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Daily Challenge — How to Fine-Tune LLMs with LoRA

**Developers Institute & Sira Labs — Week 6, Day 4**

## Objectif

Adapter le modèle pré-entraîné `bigscience/bloomz-560m` au dataset
`Abirate/english_quotes` avec **LoRA**, sans réentraîner tous ses paramètres.

Ce notebook réalise les étapes suivantes :

1. installation des bibliothèques ;
2. chargement du modèle et du tokenizer ;
3. chargement de 10 % du dataset ;
4. tokenisation des citations ;
5. configuration et application de LoRA ;
6. entraînement avec `Trainer` ;
7. sauvegarde et rechargement de l’adaptateur ;
8. génération de texte avec le modèle adapté.

> Dans Google Colab, activez un GPU avec  
> **Exécution → Modifier le type d’exécution → T4 GPU**.


In [ ]:

# Installation de versions compatibles entre elles
%pip install -q "transformers==4.48.3" "peft==0.14.0" "datasets==3.2.0" "accelerate==1.3.0"


In [ ]:

# Imports et configuration générale
import gc
import os
import time
import torch
import transformers
import peft
import datasets

from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR = "/content/cache"
OUTPUT_DIR = os.path.join(CACHE_DIR, "peft_lab_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Transformers :", transformers.__version__)
print("PEFT         :", peft.__version__)
print("Datasets     :", datasets.__version__)
print("Device       :", DEVICE)

if DEVICE.type == "cpu":
    print(
        "\nAttention : BLOOMZ-560m peut être très lent sur CPU. "
        "Un GPU T4 est recommandé."
    )



## 1. Chargement du modèle de fondation

`bigscience/bloomz-560m` est chargé comme modèle de langage causal. Le tokenizer
utilise le token de fin de séquence comme token de remplissage afin de permettre
la création de lots de tailles uniformes.


In [ ]:

model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=CACHE_DIR,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=CACHE_DIR,
)

foundation_model.config.pad_token_id = tokenizer.pad_token_id
foundation_model.config.use_cache = False

print("Modèle chargé :", model_name)
print("Pad token     :", repr(tokenizer.pad_token))
print("Taille vocab. :", len(tokenizer))


In [ ]:

# Chargement de 10 % du split d'entraînement
raw_data = load_dataset(
    "Abirate/english_quotes",
    split="train[:10%]",
    cache_dir=CACHE_DIR,
)

MAX_LENGTH = 128

def tokenize_quotes(samples):
    # Ajout d'un préfixe cohérent et du token de fin de séquence
    texts = [
        f"Quote: {quote.strip()}{tokenizer.eos_token}"
        for quote in samples["quote"]
    ]

    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_data = raw_data.map(
    tokenize_quotes,
    batched=True,
    remove_columns=raw_data.column_names,
    desc="Tokenisation des citations",
)

# Petite séparation pour suivre aussi la perte de validation
dataset_split = tokenized_data.train_test_split(
    test_size=0.10,
    seed=SEED,
)

train_data = dataset_split["train"]
eval_data = dataset_split["test"]

print("Nombre total de citations :", len(raw_data))
print("Exemples d'entraînement   :", len(train_data))
print("Exemples de validation     :", len(eval_data))

print("\nAperçu de cinq citations :")
display(
    raw_data.select(range(min(5, len(raw_data))))
    .to_pandas()[["quote", "author"]]
)


In [ ]:

# Configuration LoRA demandée dans l'exercice
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Ajout des adaptateurs LoRA au modèle de fondation
peft_model = get_peft_model(
    foundation_model,
    lora_config,
)

print("Paramètres entraînables après application de LoRA :")
peft_model.print_trainable_parameters()


In [ ]:

# Création dynamique des paramètres selon le matériel disponible
using_gpu = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    report_to="none",
    auto_find_batch_size=True,
    per_device_train_batch_size=4 if using_gpu else 1,
    per_device_eval_batch_size=4 if using_gpu else 1,
    gradient_accumulation_steps=4,
    learning_rate=3e-2,
    num_train_epochs=1,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=using_gpu,
    use_cpu=not using_gpu,
    optim="adamw_torch",
    seed=SEED,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
    processing_class=tokenizer,
)

train_result = trainer.train()

print("\nEntraînement terminé.")
print("Perte finale :", train_result.training_loss)


In [ ]:

# Sauvegarde de l'adaptateur LoRA
time_now = int(time.time())
peft_model_path = os.path.join(
    OUTPUT_DIR,
    f"peft_model_{time_now}",
)

trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

print("Adaptateur LoRA sauvegardé dans :", peft_model_path)

# Libération de la mémoire avant le rechargement
del trainer
del peft_model
del foundation_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Rechargement du modèle de fondation
inference_base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=CACHE_DIR,
)

# Chargement de l'adaptateur sauvegardé
loaded_peft_model = PeftModel.from_pretrained(
    inference_base_model,
    peft_model_path,
    is_trainable=False,
)

loaded_peft_model = loaded_peft_model.to(DEVICE)
loaded_peft_model.eval()

print("L'adaptateur LoRA a été rechargé pour l'inférence.")


In [ ]:

# Génération de texte avec le modèle LoRA rechargé
prompt = "Two things are infinite: "

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    outputs = loaded_peft_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.90,
        repetition_penalty=1.10,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print("Prompt :")
print(prompt)

print("\nTexte généré :")
print(generated_text)

print(
    "\nConclusion : le modèle BLOOMZ-560m a été adapté au style des citations "
    "en entraînant uniquement les petits adaptateurs LoRA."
)
